In [1]:
TEST = False

INPUT_FN = "input_test.txt"
TEST_SOLUTION = 24

TEST_SOLUTION2 = None

if not TEST:
    INPUT_FN = "input.txt"

def printd(s):
    if TEST:
        print (s)

In [2]:
# with open(INPUT_FN) as f:
#     s = f.read()

# patterns_block, designs_block = s.split('\n\n')

# Read and split lines
with open(INPUT_FN) as f:
    lines = [l.rstrip() for l in f]

if lines[-1]=='':
    lines=lines[:-1]

In [3]:
lines

['98162,50091',
 '98162,51304',
 '97906,51304',
 '97906,52509',
 '97702,52509',
 '97702,53715',
 '97601,53715',
 '97601,54932',
 '97612,54932',
 '97612,56127',
 '97396,56127',
 '97396,57376',
 '97551,57376',
 '97551,58487',
 '96846,58487',
 '96846,59734',
 '96918,59734',
 '96918,60909',
 '96612,60909',
 '96612,62083',
 '96306,62083',
 '96306,63240',
 '95945,63240',
 '95945,64590',
 '96197,64590',
 '96197,65604',
 '95372,65604',
 '95372,66765',
 '95007,66765',
 '95007,68035',
 '94904,68035',
 '94904,68884',
 '93777,68884',
 '93777,69964',
 '93234,69964',
 '93234,71225',
 '93064,71225',
 '93064,72295',
 '92486,72295',
 '92486,73590',
 '92312,73590',
 '92312,74615',
 '91636,74615',
 '91636,75670',
 '91010,75670',
 '91010,76223',
 '89626,76223',
 '89626,77690',
 '89623,77690',
 '89623,78215',
 '88272,78215',
 '88272,79365',
 '87788,79365',
 '87788,80160',
 '86845,80160',
 '86845,81422',
 '86462,81422',
 '86462,82540',
 '85880,82540',
 '85880,83175',
 '84768,83175',
 '84768,84030',
 '83902,

In [4]:
from skimage import morphology, segmentation
import numpy as np

In [5]:
corners = []

for line0 in lines:
    tile_coord = tuple(map(int,line0.split(',')))
    
    corners.append(tile_coord)


In [6]:
corners

[(98162, 50091),
 (98162, 51304),
 (97906, 51304),
 (97906, 52509),
 (97702, 52509),
 (97702, 53715),
 (97601, 53715),
 (97601, 54932),
 (97612, 54932),
 (97612, 56127),
 (97396, 56127),
 (97396, 57376),
 (97551, 57376),
 (97551, 58487),
 (96846, 58487),
 (96846, 59734),
 (96918, 59734),
 (96918, 60909),
 (96612, 60909),
 (96612, 62083),
 (96306, 62083),
 (96306, 63240),
 (95945, 63240),
 (95945, 64590),
 (96197, 64590),
 (96197, 65604),
 (95372, 65604),
 (95372, 66765),
 (95007, 66765),
 (95007, 68035),
 (94904, 68035),
 (94904, 68884),
 (93777, 68884),
 (93777, 69964),
 (93234, 69964),
 (93234, 71225),
 (93064, 71225),
 (93064, 72295),
 (92486, 72295),
 (92486, 73590),
 (92312, 73590),
 (92312, 74615),
 (91636, 74615),
 (91636, 75670),
 (91010, 75670),
 (91010, 76223),
 (89626, 76223),
 (89626, 77690),
 (89623, 77690),
 (89623, 78215),
 (88272, 78215),
 (88272, 79365),
 (87788, 79365),
 (87788, 80160),
 (86845, 80160),
 (86845, 81422),
 (86462, 81422),
 (86462, 82540),
 (85880, 82540

In [7]:
corners_np = np.array(corners)

In [8]:
max_coords = corners_np.max(axis=0)
min_coords = corners_np.min(axis=0)

In [9]:
print(f"{min_coords=}, {max_coords=}")

min_coords=array([1640, 1750]), max_coords=array([98211, 98399])


In [10]:
def swap_if_desc(a,b):
    if b<a:
        return b,a
    return a,b

In [11]:
# floor = np.zeros( max_coords+1, dtype=bool)
# for i,c in enumerate(corners):
#     c2=corners[0]
#     if i!=len(corners)-1:
#         c2 = corners[i+1]
    
#     xstart,xend = swap_if_desc(c[0], c2[0])
#     ystart,yend = swap_if_desc(c[1], c2[1])

#     floor[xstart:xend+1, ystart:yend+1]=True


In [12]:
import matplotlib.pyplot as plt
# if TEST:
#     plt.imshow(floor)

How to fill the inside?

In [13]:
# filled = morphology.remove_small_holes(~floor, area_threshold=1)
# filled2 = morphology.remove_small_objects(filled, min_size=10)

In [14]:
import scipy as sp
# floor_filled = sp.ndimage.binary_fill_holes(floor)

Working with the full size of the floor is not pratical. It needs 9Gb

A better solution can be to assign coordinates to consecutive numbers, in some sort of transformed space. We can still use the routine to fill the shape and then check if transformed rectangle is inside.

However the area must be calculated using the real coordinates.

In [15]:
# get list of coordinates and associate with range

X_coords = [c[0] for c in corners]
Y_coords = [c[1] for c in corners]

In [16]:
X_coords

[98162,
 98162,
 97906,
 97906,
 97702,
 97702,
 97601,
 97601,
 97612,
 97612,
 97396,
 97396,
 97551,
 97551,
 96846,
 96846,
 96918,
 96918,
 96612,
 96612,
 96306,
 96306,
 95945,
 95945,
 96197,
 96197,
 95372,
 95372,
 95007,
 95007,
 94904,
 94904,
 93777,
 93777,
 93234,
 93234,
 93064,
 93064,
 92486,
 92486,
 92312,
 92312,
 91636,
 91636,
 91010,
 91010,
 89626,
 89626,
 89623,
 89623,
 88272,
 88272,
 87788,
 87788,
 86845,
 86845,
 86462,
 86462,
 85880,
 85880,
 84768,
 84768,
 83902,
 83902,
 82956,
 82956,
 81867,
 81867,
 81349,
 81349,
 80139,
 80139,
 79207,
 79207,
 78112,
 78112,
 77184,
 77184,
 76085,
 76085,
 75433,
 75433,
 74203,
 74203,
 73118,
 73118,
 72278,
 72278,
 71065,
 71065,
 69863,
 69863,
 68694,
 68694,
 67633,
 67633,
 66593,
 66593,
 65381,
 65381,
 64372,
 64372,
 63137,
 63137,
 61874,
 61874,
 60825,
 60825,
 59502,
 59502,
 58351,
 58351,
 57169,
 57169,
 56010,
 56010,
 54811,
 54811,
 53545,
 53545,
 52338,
 52338,
 51130,
 51130,
 49909,


In [17]:
Y_coords

[50091,
 51304,
 51304,
 52509,
 52509,
 53715,
 53715,
 54932,
 54932,
 56127,
 56127,
 57376,
 57376,
 58487,
 58487,
 59734,
 59734,
 60909,
 60909,
 62083,
 62083,
 63240,
 63240,
 64590,
 64590,
 65604,
 65604,
 66765,
 66765,
 68035,
 68035,
 68884,
 68884,
 69964,
 69964,
 71225,
 71225,
 72295,
 72295,
 73590,
 73590,
 74615,
 74615,
 75670,
 75670,
 76223,
 76223,
 77690,
 77690,
 78215,
 78215,
 79365,
 79365,
 80160,
 80160,
 81422,
 81422,
 82540,
 82540,
 83175,
 83175,
 84030,
 84030,
 84802,
 84802,
 85407,
 85407,
 86657,
 86657,
 87106,
 87106,
 87881,
 87881,
 88436,
 88436,
 89214,
 89214,
 89743,
 89743,
 90976,
 90976,
 91294,
 91294,
 91838,
 91838,
 92846,
 92846,
 93152,
 93152,
 93447,
 93447,
 93791,
 93791,
 94387,
 94387,
 95066,
 95066,
 95279,
 95279,
 96114,
 96114,
 96243,
 96243,
 96220,
 96220,
 97054,
 97054,
 96687,
 96687,
 97110,
 97110,
 97398,
 97398,
 97913,
 97913,
 98217,
 98217,
 97755,
 97755,
 97898,
 97898,
 98222,
 98222,
 97652,
 97652,


In [18]:
all_coords = sorted(list(set(X_coords+Y_coords)))
all_coords

[1640,
 1750,
 1798,
 1823,
 1828,
 1906,
 1966,
 1993,
 1997,
 2022,
 2038,
 2135,
 2151,
 2190,
 2207,
 2246,
 2293,
 2323,
 2428,
 2465,
 2473,
 2522,
 2625,
 2657,
 2664,
 2666,
 2675,
 2817,
 2852,
 2869,
 2950,
 3056,
 3107,
 3147,
 3156,
 3225,
 3311,
 3368,
 3550,
 3593,
 3706,
 3708,
 3756,
 3808,
 3911,
 4189,
 4240,
 4251,
 4518,
 4576,
 4715,
 4750,
 4834,
 4990,
 5086,
 5266,
 5288,
 5442,
 5484,
 5633,
 5718,
 5792,
 6031,
 6235,
 6353,
 6485,
 6666,
 6677,
 6715,
 6752,
 7027,
 7069,
 7422,
 7647,
 7681,
 7687,
 7789,
 8219,
 8233,
 8338,
 8682,
 8747,
 8876,
 8904,
 9488,
 9505,
 9531,
 9678,
 9692,
 9733,
 9911,
 10165,
 10338,
 10629,
 10636,
 10713,
 11379,
 11593,
 11644,
 11758,
 12011,
 12171,
 12192,
 12428,
 12493,
 12624,
 12697,
 13199,
 13231,
 13808,
 13966,
 13982,
 14109,
 14187,
 14206,
 14240,
 14953,
 15160,
 15309,
 15414,
 15843,
 15972,
 15978,
 16082,
 16559,
 17002,
 17218,
 17248,
 17450,
 17721,
 17767,
 17952,
 18528,
 18729,
 18802,
 18961,
 19

In [19]:
max(all_coords)

98399

In [20]:
def get_tr_coord(coord_pair):
    global all_coords
    i0 = all_coords.index(coord_pair[0])
    i1 = all_coords.index(coord_pair[1])
    return (i0,i1)

Convert all corners to the transformed version

In [21]:
corners_tr = list(map(get_tr_coord, corners))

In [22]:
corners_tr

[(487, 246),
 (487, 249),
 (483, 249),
 (483, 253),
 (479, 253),
 (479, 258),
 (475, 258),
 (475, 262),
 (476, 262),
 (476, 266),
 (467, 266),
 (467, 269),
 (472, 269),
 (472, 273),
 (459, 273),
 (459, 278),
 (460, 278),
 (460, 281),
 (454, 281),
 (454, 285),
 (452, 285),
 (452, 289),
 (445, 289),
 (445, 293),
 (447, 293),
 (447, 298),
 (442, 298),
 (442, 302),
 (436, 302),
 (436, 305),
 (433, 305),
 (433, 310),
 (426, 310),
 (426, 313),
 (421, 313),
 (421, 318),
 (419, 318),
 (419, 321),
 (416, 321),
 (416, 326),
 (415, 326),
 (415, 330),
 (410, 330),
 (410, 334),
 (405, 334),
 (405, 337),
 (398, 337),
 (398, 341),
 (397, 341),
 (397, 346),
 (391, 346),
 (391, 349),
 (387, 349),
 (387, 352),
 (383, 352),
 (383, 358),
 (380, 358),
 (380, 362),
 (378, 362),
 (378, 366),
 (373, 366),
 (373, 369),
 (368, 369),
 (368, 374),
 (363, 374),
 (363, 376),
 (359, 376),
 (359, 382),
 (357, 382),
 (357, 385),
 (351, 385),
 (351, 388),
 (347, 388),
 (347, 392),
 (344, 392),
 (344, 395),
 (339, 395),

In [23]:
floor_tr = np.zeros( ( len(all_coords),len(all_coords) ), dtype=bool)

for i,c in enumerate(corners_tr):
    c2=corners_tr[0]
    if i!=len(corners_tr)-1:
        c2 = corners_tr[i+1]

    xstart,xend = swap_if_desc(c[0], c2[0])
    ystart,yend = swap_if_desc(c[1], c2[1])

    floor_tr[xstart:xend+1, ystart:yend+1]=True

In [24]:
floor_tr_filled = sp.ndimage.binary_fill_holes(floor_tr)

In [25]:
if TEST:
    plt.imshow(floor_tr_filled)

ok

Now, get pairs of corners and check the rectangles they define are all inside the filled floor,
and then collect its size

In [26]:
# max_area = 0
# for i in range(len(corners)-1):
#     c=corners[i]
#     for j in range(i+1,len(corners)):
#         c2=corners[j]

#         xstart,xend = swap_if_desc(c[0], c2[0])
#         ystart,yend = swap_if_desc(c[1], c2[1])

#         # check region is all floor

#         portion = floor_filled[xstart:xend+1, ystart:yend+1]
#         if np.all(portion):
#             # get the area
#             area = portion.size
#             max_area=max(max_area,area)


In [27]:
max_area = 0
for i in range(len(corners_tr)-1):
    c=corners_tr[i]
    for j in range(i+1,len(corners)):
        c2=corners_tr[j]
        
        xstart,xend = swap_if_desc(c[0], c2[0])
        ystart,yend = swap_if_desc(c[1], c2[1])

        # check region is all floor

        portion = floor_tr_filled[xstart:xend+1, ystart:yend+1]
        if np.all(portion):
            # get the area
            
            # To calculate area we need to convert the transformed corner to real
            # No need to convert, just use indices of non-transformed
            c_nt = corners[i]
            c2_nt = corners[j]

            area = (abs(c_nt[0]-c2_nt[0])+1) * (abs(c_nt[1]-c2_nt[1])+1)
            max_area=max(max_area,area)

In [28]:
if TEST:
    assert max_area == TEST_SOLUTION
max_area

1566935900